# 🌿 AgroCure — Plant Disease Detection Model
**MobileNetV2 Transfer Learning | PlantVillage Dataset | 10 Classes**

### Before running:
1. Runtime → Change runtime type → **T4 GPU**
2. Run all cells top to bottom
3. Final cell saves `agrocure_model.h5` to Google Drive

**Classes trained:**
- Tomato: Early Blight, Late Blight, Bacterial Spot, Healthy
- Potato: Early Blight, Late Blight, Healthy
- Corn: Common Rust, Northern Leaf Blight, Healthy

In [ ]:
# ── CELL 1: Install & imports ──────────────────────────────────────────────
import os
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout
from tensorflow.keras.models import Model
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.optimizers import Adam

print(f'TensorFlow version: {tf.__version__}')
print(f'GPU available: {len(tf.config.list_physical_devices("GPU")) > 0}')

In [ ]:
# ── CELL 2: Mount Google Drive ─────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

# Create output folder in Drive
os.makedirs('/content/drive/MyDrive/AgroCure', exist_ok=True)
print('Drive mounted. Model will be saved to /content/drive/MyDrive/AgroCure/')

In [ ]:
# ── CELL 3: Download PlantVillage dataset from Kaggle ──────────────────────
# Option A: Upload kaggle.json first (recommended)
# Go to kaggle.com → Account → Create API Token → upload kaggle.json here

from google.colab import files
print('Upload your kaggle.json file:')
uploaded = files.upload()  # Upload kaggle.json when prompted

# Setup kaggle credentials
os.makedirs('/root/.kaggle', exist_ok=True)
os.rename('kaggle.json', '/root/.kaggle/kaggle.json')
os.chmod('/root/.kaggle/kaggle.json', 600)

# Download PlantVillage dataset
print('Downloading PlantVillage dataset (~1.5GB)...')
!kaggle datasets download -d arjuntejaswi/plant-village
print('Download complete!')

In [ ]:
# ── CELL 4: Extract and filter only the 10 classes we need ────────────────
import zipfile
import shutil

print('Extracting dataset...')
with zipfile.ZipFile('plant-village.zip', 'r') as z:
    z.extractall('/content/plantvillage_full')

# Find where the actual folders are
for root, dirs, files_list in os.walk('/content/plantvillage_full'):
    if dirs:
        print(f'Found folders at: {root}')
        print(f'First few: {dirs[:5]}')
        break

In [ ]:
# ── CELL 5: Filter only our 10 target classes ─────────────────────────────
TARGET_CLASSES = [
    'Corn_(maize)___Common_rust_',
    'Corn_(maize)___Northern_Leaf_Blight',
    'Corn_(maize)___healthy',
    'Potato___Early_blight',
    'Potato___Late_blight',
    'Potato___healthy',
    'Tomato___Bacterial_spot',
    'Tomato___Early_blight',
    'Tomato___Late_blight',
    'Tomato___healthy',
]

# Auto-detect source folder
SRC = None
for root, dirs, _ in os.walk('/content/plantvillage_full'):
    for d in dirs:
        if 'Tomato___healthy' in dirs or 'Tomato___Late_blight' in dirs:
            SRC = root
            break
    if SRC:
        break

if SRC is None:
    # Try common paths
    for candidate in [
        '/content/plantvillage_full/PlantVillage',
        '/content/plantvillage_full/plant-village',
        '/content/plantvillage_full',
    ]:
        if os.path.isdir(candidate):
            listed = os.listdir(candidate)
            if any('Tomato' in x for x in listed):
                SRC = candidate
                break

print(f'Source folder detected: {SRC}')
available = os.listdir(SRC)
print(f'Total classes available: {len(available)}')

# Check all target classes exist
missing = [c for c in TARGET_CLASSES if c not in available]
if missing:
    print(f'WARNING - Missing classes: {missing}')
    print('Available similar classes:')
    for m in missing:
        partial = m.split('___')[0]
        matches = [a for a in available if partial in a]
        print(f'  {m} → {matches}')
else:
    print('All 10 target classes found!')

In [ ]:
# ── CELL 6: Build filtered dataset with train/val split ───────────────────
import random
from PIL import Image

DATASET_DIR = '/content/agrocure_dataset'
TRAIN_DIR = f'{DATASET_DIR}/train'
VAL_DIR = f'{DATASET_DIR}/val'
SPLIT = 0.8  # 80% train, 20% val

os.makedirs(TRAIN_DIR, exist_ok=True)
os.makedirs(VAL_DIR, exist_ok=True)

total_train, total_val = 0, 0

for cls in TARGET_CLASSES:
    src_cls = os.path.join(SRC, cls)
    if not os.path.isdir(src_cls):
        print(f'SKIP (not found): {cls}')
        continue

    images = [f for f in os.listdir(src_cls) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
    random.shuffle(images)

    split_idx = int(len(images) * SPLIT)
    train_imgs = images[:split_idx]
    val_imgs = images[split_idx:]

    os.makedirs(f'{TRAIN_DIR}/{cls}', exist_ok=True)
    os.makedirs(f'{VAL_DIR}/{cls}', exist_ok=True)

    for img in train_imgs:
        shutil.copy(os.path.join(src_cls, img), f'{TRAIN_DIR}/{cls}/{img}')
    for img in val_imgs:
        shutil.copy(os.path.join(src_cls, img), f'{VAL_DIR}/{cls}/{img}')

    total_train += len(train_imgs)
    total_val += len(val_imgs)
    print(f'{cls}: {len(train_imgs)} train | {len(val_imgs)} val')

print(f'\nTotal: {total_train} train | {total_val} val')

In [ ]:
# ── CELL 7: Data generators ────────────────────────────────────────────────
IMG_SIZE = (224, 224)
BATCH_SIZE = 32
NUM_CLASSES = len(TARGET_CLASSES)

# Training augmentation
train_datagen = ImageDataGenerator(
    preprocessing_function=tf.keras.applications.mobilenet_v2.preprocess_input,
    rotation_range=20,
    width_shift_range=0.15,
    height_shift_range=0.15,
    shear_range=0.1,
    zoom_range=0.15,
    horizontal_flip=True,
    fill_mode='nearest'
)

# Validation - no augmentation, just preprocess
val_datagen = ImageDataGenerator(
    preprocessing_function=tf.keras.applications.mobilenet_v2.preprocess_input
)

train_gen = train_datagen.flow_from_directory(
    TRAIN_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=True
)

val_gen = val_datagen.flow_from_directory(
    VAL_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False
)

# IMPORTANT: Print class indices so you can verify CLASS_LABELS order
print('\nClass index mapping (USE THIS ORDER in predict.py):')
class_indices = train_gen.class_indices
idx_to_class = {v: k for k, v in class_indices.items()}
for i in range(NUM_CLASSES):
    print(f'  {i}: {idx_to_class[i]}')

In [ ]:
# ── CELL 8: Build model ────────────────────────────────────────────────────
def build_model(num_classes):
    # Load MobileNetV2 pretrained on ImageNet, exclude top classifier
    base = MobileNetV2(
        weights='imagenet',
        include_top=False,
        input_shape=(224, 224, 3)
    )

    # Freeze base layers initially
    base.trainable = False

    # Add custom classifier head
    x = base.output
    x = GlobalAveragePooling2D()(x)
    x = Dense(256, activation='relu')(x)
    x = Dropout(0.3)(x)
    x = Dense(128, activation='relu')(x)
    x = Dropout(0.2)(x)
    output = Dense(num_classes, activation='softmax')(x)

    model = Model(inputs=base.input, outputs=output)
    return model, base

model, base_model = build_model(NUM_CLASSES)

model.compile(
    optimizer=Adam(learning_rate=1e-3),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

print(f'Total params: {model.count_params():,}')
print(f'Trainable params: {sum([tf.size(w).numpy() for w in model.trainable_weights]):,}')

In [ ]:
# ── CELL 9: Phase 1 Training — frozen base (fast, ~5 min) ─────────────────
callbacks = [
    ModelCheckpoint(
        '/content/best_disease_model.h5',
        monitor='val_accuracy',
        save_best_only=True,
        verbose=1
    ),
    EarlyStopping(
        monitor='val_accuracy',
        patience=5,
        restore_best_weights=True,
        verbose=1
    ),
    ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.3,
        patience=3,
        min_lr=1e-7,
        verbose=1
    )
]

print('Phase 1: Training classifier head only (base frozen)...')
history1 = model.fit(
    train_gen,
    epochs=10,
    validation_data=val_gen,
    callbacks=callbacks,
    verbose=1
)

print(f'\nPhase 1 best val accuracy: {max(history1.history["val_accuracy"]):.4f}')

In [ ]:
# ── CELL 10: Phase 2 Training — fine-tune top layers (~15 min) ────────────
# Unfreeze top 30 layers of base model for fine-tuning
base_model.trainable = True
fine_tune_from = len(base_model.layers) - 30

for layer in base_model.layers[:fine_tune_from]:
    layer.trainable = False

trainable_count = sum(1 for l in base_model.layers if l.trainable)
print(f'Unfroze top {trainable_count} base layers for fine-tuning')

model.compile(
    optimizer=Adam(learning_rate=1e-4),  # Lower LR for fine-tuning
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

print('Phase 2: Fine-tuning...')
history2 = model.fit(
    train_gen,
    epochs=15,
    validation_data=val_gen,
    callbacks=callbacks,
    verbose=1
)

print(f'\nPhase 2 best val accuracy: {max(history2.history["val_accuracy"]):.4f}')

In [ ]:
# ── CELL 11: Evaluate on validation set ───────────────────────────────────
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns

# Load best saved model
best_model = tf.keras.models.load_model('/content/best_disease_model.h5')

val_gen.reset()
loss, acc = best_model.evaluate(val_gen, verbose=0)
print(f'Final Validation Accuracy: {acc:.4f} ({acc*100:.2f}%)')
print(f'Final Validation Loss: {loss:.4f}')

# Predictions
val_gen.reset()
preds = best_model.predict(val_gen, verbose=1)
pred_classes = np.argmax(preds, axis=1)
true_classes = val_gen.classes
class_names = list(idx_to_class[i] for i in range(NUM_CLASSES))

print('\nClassification Report:')
print(classification_report(true_classes, pred_classes, target_names=class_names))

In [ ]:
# ── CELL 12: Plot training curves ─────────────────────────────────────────
# Combine both phases
all_acc = history1.history['accuracy'] + history2.history['accuracy']
all_val_acc = history1.history['val_accuracy'] + history2.history['val_accuracy']
all_loss = history1.history['loss'] + history2.history['loss']
all_val_loss = history1.history['val_loss'] + history2.history['val_loss']

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(all_acc, label='Train Accuracy')
ax1.plot(all_val_acc, label='Val Accuracy')
ax1.axvline(x=len(history1.history['accuracy'])-1, color='gray', linestyle='--', label='Fine-tune start')
ax1.set_title('Model Accuracy')
ax1.set_xlabel('Epoch')
ax1.legend()

ax2.plot(all_loss, label='Train Loss')
ax2.plot(all_val_loss, label='Val Loss')
ax2.axvline(x=len(history1.history['loss'])-1, color='gray', linestyle='--', label='Fine-tune start')
ax2.set_title('Model Loss')
ax2.set_xlabel('Epoch')
ax2.legend()

plt.tight_layout()
plt.savefig('/content/drive/MyDrive/AgroCure/training_curves_disease.png', dpi=150)
plt.show()
print('Training curves saved to Drive!')

In [ ]:
# ── CELL 13: Save final model + class labels ───────────────────────────────
import json

# Save model
MODEL_SAVE_PATH = '/content/drive/MyDrive/AgroCure/agrocure_model.h5'
best_model.save(MODEL_SAVE_PATH)
print(f'Model saved to: {MODEL_SAVE_PATH}')

# Save class labels — PASTE THIS INTO predict.py CLASS_LABELS
labels_path = '/content/drive/MyDrive/AgroCure/disease_class_labels.json'
with open(labels_path, 'w') as f:
    json.dump(idx_to_class, f, indent=2)

print('\n=== COPY THIS INTO model/predict.py CLASS_LABELS ===')
print('CLASS_LABELS = [')
for i in range(NUM_CLASSES):
    print(f'    "{idx_to_class[i]}",  # index {i}')
print(']')

print(f'\nFinal model accuracy: {acc*100:.2f}%')
print('Done! Download agrocure_model.h5 from your Google Drive/AgroCure folder.')

In [ ]:
# ── CELL 14: Quick test on a sample image ─────────────────────────────────
# Optional: test with a single image
import urllib.request
from PIL import Image
import io

def predict_single(img_path):
    img = Image.open(img_path).convert('RGB').resize((224, 224))
    arr = np.array(img, dtype=np.float32)
    arr = tf.keras.applications.mobilenet_v2.preprocess_input(arr)
    arr = np.expand_dims(arr, 0)
    probs = best_model.predict(arr, verbose=0)[0]
    top3 = np.argsort(probs)[::-1][:3]
    print('Top 3 predictions:')
    for i in top3:
        print(f'  {idx_to_class[i]}: {probs[i]*100:.2f}%')

# Pick any val image to test
test_cls = os.listdir(VAL_DIR)[0]
test_imgs = os.listdir(f'{VAL_DIR}/{test_cls}')
test_img_path = f'{VAL_DIR}/{test_cls}/{test_imgs[0]}'
print(f'Testing with: {test_cls}')
predict_single(test_img_path)